# 🚀 Deep-Live-Cam with T4 GPU (Fixed)
## Enhanced Face Swap + Perfect Mouth Mask

**⚠️ Make sure GPU is enabled: Runtime → Change runtime type → T4 GPU**

In [ ]:
# Fix NumPy compatibility issue
!pip install "numpy<2.0" --force-reinstall
!pip install opencv-python==4.8.0.74 --force-reinstall

# Restart runtime
import os
os.kill(os.getpid(), 9)

In [ ]:
# Check GPU and NumPy
!nvidia-smi
import numpy as np
print(f"NumPy version: {np.__version__}")
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")

In [ ]:
# Install dependencies
!apt update -qq && apt install -y ffmpeg
!git clone https://github.com/Mayank-kanojiya/deeplive.git
%cd deeplive

!pip install insightface==0.7.3
!pip install onnxruntime-gpu==1.21.0
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install gradio pillow tqdm

In [ ]:
# Download models
import urllib.request
import os

os.makedirs('models', exist_ok=True)
model_url = "https://huggingface.co/hacksider/deep-live-cam/resolve/main/inswapper_128_fp16.onnx"
urllib.request.urlretrieve(model_url, "models/inswapper_128_fp16.onnx")
print("✅ Model downloaded")

In [ ]:
# Face swap implementation
import cv2
import numpy as np
import insightface
import gradio as gr

# Initialize
face_app = insightface.app.FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(640, 640))
face_swapper = insightface.model_zoo.get_model('models/inswapper_128_fp16.onnx', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])

def get_face(image):
    faces = face_app.get(image)
    return faces[0] if faces else None

def create_mouth_mask(face, frame):
    mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    if face is None or not hasattr(face, 'landmark_2d_106'):
        return mask, None, (0,0,0,0)
    
    landmarks = face.landmark_2d_106
    if landmarks is None or landmarks.shape[0] < 106:
        return mask, None, (0,0,0,0)
    
    # Mouth landmarks
    mouth_indices = [65, 66, 62, 70, 69, 18, 19, 20, 21, 22, 23, 24, 0, 8, 7, 6, 5, 4, 3, 2]
    mouth_landmarks = landmarks[mouth_indices].astype(np.int32)
    
    # Create mask
    cv2.fillPoly(mask, [mouth_landmarks], 255)
    mask = cv2.GaussianBlur(mask, (15, 15), 5)
    
    # Bounding box
    x, y, w, h = cv2.boundingRect(mouth_landmarks)
    padding = 10
    x, y = max(0, x-padding), max(0, y-padding)
    w, h = min(frame.shape[1]-x, w+2*padding), min(frame.shape[0]-y, h+2*padding)
    
    mouth_cutout = frame[y:y+h, x:x+w].copy()
    return mask, mouth_cutout, (x, y, x+w, y+h)

def face_swap(source_img, target_img, use_mouth_mask=True):
    source_face = get_face(source_img)
    target_face = get_face(target_img)
    
    if not source_face:
        return target_img, "❌ No face in source"
    if not target_face:
        return target_img, "❌ No face in target"
    
    # Swap
    result = face_swapper.get(target_img, target_face, source_face, paste_back=True)
    
    if use_mouth_mask:
        mask, mouth_cutout, box = create_mouth_mask(target_face, target_img)
        if mouth_cutout is not None:
            x1, y1, x2, y2 = box
            roi = result[y1:y2, x1:x2]
            if roi.shape[:2] == mouth_cutout.shape[:2]:
                mask_roi = mask[y1:y2, x1:x2] / 255.0
                blended = mouth_cutout * mask_roi[:,:,None] + roi * (1 - mask_roi[:,:,None])
                result[y1:y2, x1:x2] = blended.astype(np.uint8)
    
    return result, "✅ Success"

def process(source, target, mouth_mask):
    if source is None or target is None:
        return None, "Upload both images"
    
    source_bgr = cv2.cvtColor(source, cv2.COLOR_RGB2BGR)
    target_bgr = cv2.cvtColor(target, cv2.COLOR_RGB2BGR)
    result_bgr, status = face_swap(source_bgr, target_bgr, mouth_mask)
    result_rgb = cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB)
    return result_rgb, status

print("✅ Face swap ready")

In [ ]:
# Launch interface
with gr.Blocks(title="Face Swap T4") as demo:
    gr.Markdown("# 🚀 Face Swap with T4 GPU")
    
    with gr.Row():
        source = gr.Image(label="Source Face", type="numpy")
        target = gr.Image(label="Target Image", type="numpy")
        result = gr.Image(label="Result")
    
    with gr.Row():
        mouth_mask = gr.Checkbox(label="Enable Mouth Mask", value=True)
        swap_btn = gr.Button("Swap Faces", variant="primary")
    
    status = gr.Textbox(label="Status")
    
    swap_btn.click(process, [source, target, mouth_mask], [result, status])

demo.launch(share=True)